# MSD Manuals Scraper – All-in-One Notebook

**Pipeline 4 buoc:**
1. Discovery & Mapping (lap ban do URL)
2. Scrape -> Bronze Layer (extraction + context-aware chunking)
3. Silver Layer (NER + Graph Edges + Table->PNG)
4. Gold Export (Qdrant + KG + ColPali)

> **Resume-safe:** Dung bat cu luc nao, chay lai tiep tuc tu cho dung.

In [ ]:
!pip install -q requests beautifulsoup4 html2image spacy lxml
!python -m spacy download xx_ent_wiki_sm -q
print("DONE install")

## Cell 2 – Config + Helpers (chay 1 lan dau)

In [ ]:
import requests, json, time, random, re, hashlib
from bs4 import BeautifulSoup, Tag
from pathlib import Path
from datetime import datetime

BASE_URL    = "https://www.msdmanuals.com"
START_PATHS = ["/vi/chuyen-gia"]
QUEUE_FILE  = Path("url_queue.json")
BRONZE_DIR  = Path("bronze"); BRONZE_DIR.mkdir(exist_ok=True)
SILVER_DIR  = Path("silver"); SILVER_DIR.mkdir(exist_ok=True)
TABLE_DIR   = Path("silver/table_images"); TABLE_DIR.mkdir(exist_ok=True)
GOLD_DIR    = Path("gold");   GOLD_DIR.mkdir(exist_ok=True)
DELAY       = (2.0, 5.0)
HEADERS     = {
    "User-Agent": "MRAG-Medical-Research-Bot/1.0 (Academic Project HUTECH)",
    "Accept-Language": "vi-VN,vi;q=0.9"
}

def fetch(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"[FAIL] {url}: {e}"); return None

def sleep(): time.sleep(random.uniform(*DELAY))

def load_queue():
    if QUEUE_FILE.exists():
        return json.loads(QUEUE_FILE.read_text(encoding="utf-8"))
    return {"pending":[], "done":[], "failed":[], "created": str(datetime.now())}

def save_queue(q):
    QUEUE_FILE.write_text(json.dumps(q, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  queue: pending={len(q['pending'])}  done={len(q['done'])}  failed={len(q['failed'])}")

def url_to_fname(url):
    slug = re.sub(r"[^\w\-]", "_", url.replace("https://www.msdmanuals.com",""))
    return slug[:180] + ".json"

def table_to_md(table):
    rows = []
    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True) for td in tr.find_all(["th","td"])]
        rows.append("| " + " | ".join(cells) + " |")
    if len(rows) > 1:
        cols = len(rows[0].split("|")) - 2
        rows.insert(1, "|" + " --- |"*cols)
    return "\n".join(rows)

DRUG_RE    = re.compile(r"\b(Warfarin|Aspirin|Metformin|Amoxicillin|Atorvastatin|Paracetamol|Ibuprofen|Furosemide|Digoxin|Amlodipine|Metoprolol|Simvastatin|Codein|Morphine|Insulin)\b", re.I)
DISEASE_RE = re.compile(r"\b(tang huyet ap|suy tim|tieu duong|ung thu|nhiem khuan|viem phoi|dot quy|nhoi mau co tim|suy than)\b", re.I)

def ner(text):
    return {"drugs": list(set(DRUG_RE.findall(text))), "diseases": list(set(DISEASE_RE.findall(text)))}

def rel_type(ctx):
    c = ctx.lower()
    if any(k in c for k in ["tuong tac","interaction"]): return "INTERACTS_WITH"
    if any(k in c for k in ["chong chi dinh","contraindication"]): return "CONTRAINDICATED_IN"
    if any(k in c for k in ["tac dung phu","adverse"]): return "HAS_SIDE_EFFECT"
    return None

NOISE = ["nav","footer","header","aside","script","style"]

def extract_blocks(soup):
    for tag in NOISE:
        for el in soup.find_all(tag): el.decompose()
    article = soup.find("article") or soup.find("main") or soup.body
    if not article: return []
    blocks, hstack = [], []
    for el in article.descendants:
        if not isinstance(el, Tag): continue
        if el.name in ["h1","h2","h3","h4"]:
            level = int(el.name[1]); text = el.get_text(strip=True)
            hstack = [(l,t) for l,t in hstack if l < level]; hstack.append((level,text))
        elif el.name == "p" and el.parent.name not in ["td","th","li"]:
            t = el.get_text(" ", strip=True)
            if len(t) > 30: blocks.append({"heading_path":[t for _,t in hstack],"type":"paragraph","content":t})
        elif el.name == "table":
            mdx = table_to_md(el)
            if mdx: blocks.append({"heading_path":[t for _,t in hstack],"type":"table_markdown","content":mdx,"table_html":str(el)})
        elif el.name == "li" and el.parent.name in ["ul","ol"]:
            t = el.get_text(" ", strip=True)
            if len(t) > 20: blocks.append({"heading_path":[t for _,t in hstack],"type":"list_item","content":t})
    return blocks

def make_chunks(blocks, title, url, max_w=800):
    chunks, cur, cur_path = [], "", []
    def flush(path, text, ctype="text"):
        if text.strip():
            cid = hashlib.md5((url+text[:50]).encode()).hexdigest()[:12]
            chunks.append({"chunk_id":cid,"source_url":url,"source_title":title,
                           "context_path":" > ".join(path),"content":text.strip(),
                           "type":ctype,"word_count":len(text.split())})
    for b in blocks:
        if b["type"]=="table_markdown":
            flush(cur_path,cur); cur=""
            flush(b["heading_path"],b["content"],"table"); continue
        if b["heading_path"]!=cur_path and cur:
            flush(cur_path,cur); cur=""
        cur_path = b["heading_path"]; cur += " "+b["content"]
        if len(cur.split()) > max_w: flush(cur_path,cur); cur=""
    flush(cur_path,cur)
    return chunks

print("Config + Helpers OK")

## Cell 3 – Buoc 1: Discovery URLs

In [ ]:
# ── BUOC 1: Discovery URLs tu Sitemap XML ─────────────────────────────────────
# MSD Manuals co sitemap XML chinh thuc: 2607 bai viet tieng Viet
# Luu y: url_queue.json co the da co san neu ban da chay get_sitemap truoc

SITEMAP_URL = "https://www.msdmanuals.com/vi/sitemaps/professional-topic.xml.gz"

if QUEUE_FILE.exists():
    q = load_queue()
    print(f"[RESUME] url_queue.json da co: {len(q['pending'])} pending, {len(q['done'])} done")
else:
    print("[DOWNLOAD] Dang tai sitemap tu MSD Manuals...")
    r = requests.get(SITEMAP_URL, headers=HEADERS, timeout=30)
    try:
        soup = BeautifulSoup(r.content, "xml")
        urls = [loc.text for loc in soup.find_all("loc")]
    except Exception:
        soup = BeautifulSoup(r.text, "html.parser")
        urls = [loc.text for loc in soup.find_all("loc")]
    q = {"pending": urls, "done": [], "failed": [], "created": str(datetime.now())}
    save_queue(q)
    print(f"[DONE] Da luu {len(urls)} URLs vao url_queue.json")

print(f"San sang cao: {len(q['pending'])} bai | Da cao: {len(q['done'])} | Loi: {len(q['failed'])}")

## Cell 4 – Buoc 2: Scrape -> Bronze Layer

> Doi `MAX_PER_RUN = 50` -> `500` neu muon cao nhieu hon.

In [ ]:
# ── BUOC 2: Scrape -> Bronze Layer ────────────────────────────────────────────
MAX_PER_RUN = 50   # Tang len de cao nhieu hon moi lan chay

q = load_queue()
processed = 0
while q["pending"] and processed < MAX_PER_RUN:
    url = q["pending"].pop(0)
    fname = BRONZE_DIR / url_to_fname(url)
    if fname.exists():
        q["done"].append(url); continue
    print(f"[{processed+1}/{MAX_PER_RUN}] {url}")
    soup = fetch(url)
    if not soup:
        q["failed"].append(url); save_queue(q); sleep(); continue
    h1 = soup.find("h1")
    upd = soup.find(attrs={"class": re.compile(r"last.?updated|date",re.I)})
    meta = {"url":url,"scraped_at":str(datetime.now()),
            "title": h1.get_text(strip=True) if h1 else "",
            "last_updated": upd.get_text(strip=True) if upd else "",
            "authors":[]}
    blocks = extract_blocks(soup)
    chunks = make_chunks(blocks, meta["title"], url)
    fname.write_text(json.dumps({"metadata":meta,"raw_blocks":blocks,"chunks":chunks},
                                ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  -> {len(chunks)} chunks | con lai: {len(q['pending'])}")
    q["done"].append(url)
    save_queue(q); sleep(); processed += 1

print(f"[DONE] Da xu ly {processed} URL | Bronze files: {len(list(BRONZE_DIR.glob('*.json')))}")

## Cell 5 – Buoc 3: Silver Processing

In [ ]:
# ── BUOC 3: Silver Processing (NER + Graph Edges + Table->Image) ──────────────
for bf in BRONZE_DIR.glob("*.json"):
    sf = SILVER_DIR / bf.name
    if sf.exists(): print(f"[SKIP] {bf.name}"); continue

    data   = json.loads(bf.read_text(encoding="utf-8"))
    meta   = data["metadata"]
    silver_chunks, edges = [], []

    for chunk in data["chunks"]:
        sc = dict(chunk)
        sc["entities"]      = ner(chunk["content"])
        sc["relation_type"] = rel_type(chunk["context_path"])

        # Graph edge suggestion
        if sc["relation_type"] and sc["entities"]["drugs"]:
            title_drugs = DRUG_RE.findall(meta["title"])
            fallback_source = meta.get("title", "UNKNOWN").strip()
            for drug in sc["entities"]["drugs"]:
                for src in (title_drugs or [fallback_source]):
                    if src.lower() != drug.lower():
                        edges.append({"source":src,"target":drug,
                                  "relation":sc["relation_type"],
                                  "evidence_url":meta["url"],
                                  "evidence_text":chunk["content"][:200]})

        # Table -> PNG (phuc vu ColPali training)
        sc["table_image_path"] = None
        if chunk["type"] == "table":
            raw_match = [b for b in data["raw_blocks"]
                         if b["type"]=="table_markdown" and b["content"]==chunk["content"]]
            if raw_match and "table_html" in raw_match[0]:
                try:
                    from html2image import Html2Image
                    hti = Html2Image(output_path=str(TABLE_DIR))
                    img_f = f"table_{chunk['chunk_id']}.png"
                    table_html = raw_match[0]["table_html"]
                    full_html = "<html><body style='background:#fff;padding:10px;font-family:Arial'>" + table_html + "</body></html>"
                    hti.screenshot(html_str=full_html, save_as=img_f)
                    sc["table_image_path"] = str(TABLE_DIR / img_f)
                except Exception as e:
                    print(f"  [WARN] table render: {e}")

        silver_chunks.append(sc)

    sf.write_text(json.dumps({"metadata":meta,"chunks":silver_chunks,
                               "graph_edges_suggested":edges,
                               "stats":{"chunks":len(silver_chunks),"edges":len(edges)}},
                              ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  [OK] {bf.name}: {len(silver_chunks)} chunks, {len(edges)} graph edges")

print("[DONE] Silver processing hoan tat")

## Cell 6 – Buoc 4: Gold Export

In [ ]:
# ── BUOC 4: Gold Export (chunks.jsonl + graph_edges.json + colpali_pairs) ─────
all_chunks, all_edges = [], []

for sf in SILVER_DIR.glob("*.json"):
    if sf.name.startswith("table_"): continue
    data = json.loads(sf.read_text(encoding="utf-8"))
    meta = data["metadata"]
    for chunk in data["chunks"]:
        all_chunks.append({
            "chunk_id":     chunk["chunk_id"],
            "content":      chunk["content"],
            "context_path": chunk["context_path"],
            "source_url":   chunk["source_url"],
            "source_title": chunk["source_title"],
            "type":         chunk["type"],
            "entities":     chunk.get("entities",{}),
            "relation_type":chunk.get("relation_type"),
            "table_image":  chunk.get("table_image_path"),
            "payload": {"source":"msd_manuals","lang":"vi",
                        "last_updated":meta.get("last_updated",""),
                        "authors":meta.get("authors",[])}
        })
    all_edges.extend(data.get("graph_edges_suggested",[]))

# 1. Text chunks -> Qdrant
with open(GOLD_DIR/"chunks.jsonl","w",encoding="utf-8") as f:
    for c in all_chunks: f.write(json.dumps(c,ensure_ascii=False)+"\n")

# 2. Graph edges -> Neo4j / NetworkX
(GOLD_DIR/"graph_edges.json").write_text(
    json.dumps(all_edges,ensure_ascii=False,indent=2), encoding="utf-8")

# 3. ColPali table pairs -> train.jsonl
cp = [{"query": c["context_path"]+": "+c["content"][:100],
       "image_path": c["table_image"]}
      for c in all_chunks if c["type"]=="table" and c.get("table_image")]
with open(GOLD_DIR/"colpali_table_pairs.jsonl","w",encoding="utf-8") as f:
    for p in cp: f.write(json.dumps(p,ensure_ascii=False)+"\n")

print(f"""
GOLD SUMMARY
  Text chunks  (-> Qdrant):         {len(all_chunks)}
  Graph edges  (-> KG):             {len(all_edges)}
  ColPali pairs(-> Train ColPali):  {len(cp)}

Files:
  gold/chunks.jsonl
  gold/graph_edges.json
  gold/colpali_table_pairs.jsonl
""")